# S39_14 — Transformers & HuggingFace (2026 NLP Standard)

## Why Transformers replaced NLTK-era NLP

The classical NLP pipeline (tokenise → TF-IDF → logistic regression) works but has a fundamental limitation: it treats words as independent tokens. It can't capture *meaning*, *context*, or *word order* beyond n-grams.

**Transformers** (Vaswani et al., 2017 — "Attention Is All You Need") solved this with the **self-attention mechanism**: every token attends to every other token in the sequence, capturing long-range dependencies and contextual meaning. BERT (2018), GPT-2 (2019), and their successors fine-tuned on billions of text tokens now encode richer representations than any hand-crafted feature.

## Installation

```bash
pip install transformers datasets torch
```

## 1. The `pipeline` API — fastest way to get results

In [ ]:
from transformers import pipeline

# Sentiment analysis (fine-tuned DistilBERT by default)
sentiment = pipeline('sentiment-analysis')
print(sentiment('The movie was absolutely brilliant!'))
# [{'label': 'POSITIVE', 'score': 0.9998}]

# Zero-shot classification — no fine-tuning needed
classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')
result = classifier(
    'The new vaccine shows 95% efficacy in clinical trials.',
    candidate_labels=['medicine', 'politics', 'technology', 'sports']
)
print(result['labels'][0], result['scores'][0])

# Named Entity Recognition
ner = pipeline('ner', grouped_entities=True)
print(ner('Elon Musk founded SpaceX in Hawthorne, California.'))

## 2. Text classification — fine-tuning DistilBERT

Fine-tuning freezes the transformer body (pretrained weights) and trains a classification head on your labelled data.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score

# Load a dataset from HuggingFace Hub
dataset = load_dataset('imdb')  # 25k train, 25k test

model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding=True, max_length=512)

tokenized = dataset.map(tokenize, batched=True, batch_size=64)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {'accuracy': accuracy_score(labels, preds)}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'].select(range(2000)),  # subset for demo
    eval_dataset=tokenized['test'].select(range(500)),
    compute_metrics=compute_metrics,
)

trainer.train()

## 3. Sentence embeddings with `sentence-transformers`

Encodes text into dense vectors — use for semantic search, clustering, and similarity.

In [ ]:
# pip install sentence-transformers
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')  # fast, 384-dim embeddings

sentences = [
    'How do I install Python?',
    'What is the installation process for Python?',
    'How do I make pasta?',
]

embeddings = model.encode(sentences)
sim = cosine_similarity(embeddings)

print(f'Q1 vs Q2 similarity: {sim[0,1]:.3f}')  # ~0.93 — same question
print(f'Q1 vs Q3 similarity: {sim[0,2]:.3f}')  # ~0.15 — unrelated

## 4. spaCy — production NLP pipelines

spaCy is the industrial-strength alternative to NLTK — faster, more accurate, and designed for production.

In [ ]:
# pip install spacy && python -m spacy download en_core_web_sm
import spacy

nlp = spacy.load('en_core_web_sm')
doc = nlp('Apple Inc. was founded by Steve Jobs in Cupertino, California in 1976.')

print('Entities:')
for ent in doc.ents:
    print(f'  {ent.text:20} {ent.label_}')

print('\nTokens (lemma, POS):')
for token in doc[:5]:
    print(f'  {token.text:12} {token.lemma_:12} {token.pos_}')

## Quick reference

| Task | Library | Model / Function |
|------|---------|------------------|
| Sentiment analysis | `transformers` | `pipeline('sentiment-analysis')` |
| Text classification (fine-tune) | `transformers` | `AutoModelForSequenceClassification` |
| NER | `transformers` / `spaCy` | `pipeline('ner')` / `doc.ents` |
| Sentence embeddings | `sentence-transformers` | `SentenceTransformer` |
| Semantic search | `sentence-transformers` + vector DB | `encode` → cosine similarity |
| Zero-shot classification | `transformers` | `pipeline('zero-shot-classification')` |
| Production NLP pipelines | `spaCy` | `nlp.pipe()` |
| Token-level tasks (POS, lemma) | `spaCy` | `nlp(text)` |

**Further reading:**
- [HuggingFace course](https://huggingface.co/learn/nlp-course) — free, comprehensive
- [spaCy documentation](https://spacy.io/usage)
- [sentence-transformers](https://www.sbert.net/)
